# 13 - File Handling


## 1. File Handling Fundamentals (`open`, Paths & Modes)

Python'da dosya işlemleri, yerel veya uzak dizinlerdeki metin, JSON ve ikili (binary) dosyaları okumak, oluşturmak ve değiştirmek için yerleşik `open()` fonksiyonu ve dosya yolları (relative/absolute paths) üzerinden yürütülür.

**Syntax**
```python
open(file, mode='r', encoding='utf-8')
```

**Temel Modlar ve Türler**
* **Erişim Modları:** `'r'` (okuma - varsayılan), `'w'` (üzerine yazma/sıfırlama), `'a'` (sonuna ekleme - append), `'x'` (yalnızca yoksa oluşturma - exclusive creation) ve `'+'` (okuma + yazma birleşimi: `'r+'`, `'w+'`, `'a+'`, `'x+'`).
* **Veri Türü Modları:** `'t'` (metin modu - varsayılan, örn. `'rt'`) ve `'b'` (ikili/byte modu - görseller, ses dosyaları ve serileştirilmiş nesneler için, örn. `'rb'`, `'wb'`).
* **Dikkat Edilmesi Gerekenler:** Göreceli yollar (relative paths) betiğin çalıştığı mevcut çalışma dizinine (CWD) göre çözümlenir; çalışma dizini değiştiğinde dosya bulunamaması (`FileNotFoundError`) riskine karşı modern Python'da `pathlib.Path` kullanımı tercih edilir.


In [ ]:
from pathlib import Path

relative_path: Path = Path('sample_note.txt')
absolute_path: Path = relative_path.resolve()

print(f'Relative: {relative_path}')
print(f'Absolute: {absolute_path}')


## 2. Reading Files (`with`, `read`, `readline`, `readlines`)

Dosya açma işlemlerinde `with` bağlam yöneticisi (context manager) kullanmak, blok tamamlandığında veya bir hata oluştuğunda dosya tanıtıcısının (`file.close()`) otomatik ve güvenli şekilde kapatılmasını sağlar.

**Syntax**
```python
with open('file.txt', mode='r', encoding='utf-8') as f:
    content = f.read()
    line = f.readline()
    lines = f.readlines()
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **`read(size=-1)`:** Tüm dosyayı veya belirtilen karakter sayısı kadarını tek bir `str` olarak döndürür.
* **`readline()`:** Dosya imlecinin bulunduğu konumdan itibaren tek bir satır okur; dosya sonuna (EOF) ulaşıldığında boş dize (`''`) döndürür.
* **`readlines()`:** Kalan tüm satırları satır sonu karakterleriyle (`\n`) birlikte bir `list[str]` içine yükler. Büyük dosyalarda tüm listeyi belleğe almak yerine doğrudan `for line in f:` döngüsüyle satır satır yineleme yapılmalıdır.


In [ ]:
from pathlib import Path

sample_file: Path = Path('sample_read.txt')
sample_file.write_text('First line\nSecond line\nThird line\n', encoding='utf-8')

with open(sample_file, mode='r', encoding='utf-8') as f:
    full_text: str = f.read()
    print(full_text.strip())

with open(sample_file, mode='r', encoding='utf-8') as f:
    first_five: str = f.read(5)
    rest_of_first_line: str = f.readline()
    remaining_lines: list[str] = f.readlines()

print(repr(first_five))
print(repr(rest_of_first_line))
print(remaining_lines)
print(f.closed)


## 3. Writing & Creating Files (`w`, `a`, `x`, `+`, `seek`)

Dosyalara yeni veri eklemek, mevcut içeriği tamamen değiştirmek veya yalnızca dosya mevcut değilse oluşturmak için yazma ve oluşturma modları kullanılır.

**Syntax**
```python
with open('file.txt', mode='w+', encoding='utf-8') as f:
    f.write('Hello\n')
    f.seek(0)
    data = f.read()
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **`'a'` (Append):** İmleci dosya sonuna konumlandırır; mevcut içeriği koruyarak ekleme yapar ve yeni satır (`\n`) karakterini otomatik eklemez.
* **`'w'` (Write):** Dosya varsa mevcut içeriğini anında siler (truncate), yoksa yeni dosya oluşturur.
* **`'x'` (Exclusive Creation):** Dosya zaten mevcutsa `FileExistsError` fırlatarak yanlışlıkla üzerine yazmayı önler.
* **`seek(offset)`:** `'a+'`, `'w+'` veya `'x+'` modlarında yazma işlemi bittikten sonra imleç dosya sonunda kalır; yazdığınız veriyi hemen okuyabilmek için `f.seek(0)` ile imleci başa almanız gerekir.


In [ ]:
from pathlib import Path

log_file: Path = Path('app_events.txt')
log_file.unlink(missing_ok=True)

with open(log_file, mode='x+', encoding='utf-8') as f:
    f.write('Session started\n')
    f.seek(0)
    print(f.read().strip())

with open(log_file, mode='a+', encoding='utf-8') as f:
    f.write('User authenticated\n')
    f.seek(0)
    print(f.read().strip())

with open(log_file, mode='w+', encoding='utf-8') as f:
    f.write('Overwritten content\n')
    f.seek(0)
    print(f.read().strip())


## 4. Deleting Files & Directories (`os.remove`, `os.rmdir`, `pathlib`)

Dosya sistemindeki tekil dosyaları veya boş klasörleri programatik olarak silmek için standart kütüphanedeki `os` veya modern `pathlib` modülleri kullanılır.

**Syntax**
```python
import os
from pathlib import Path

if os.path.exists('file.txt'):
    os.remove('file.txt')
os.rmdir('empty_dir')

Path('file.txt').unlink(missing_ok=True)
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Kalıcı Silme Uyarısı:** `os.remove()` ve `Path.unlink()` dosyaları geri dönüşüm kutusuna göndermez, doğrudan dosya sisteminden kalıcı olarak siler.
* **Klasör Silme Kısıtı:** `os.rmdir()` ve `Path.rmdir()` yalnızca **tamamen boş** klasörleri silebilir; içinde dosya varsa `OSError` fırlatır. Dolu bir dizin ağacını tek seferde silmek için `shutil.rmtree()` kullanılır.
* **Güvenli Silme:** Silme öncesinde varlık kontrolü yapmak veya Python 3.8+ ile gelen `Path.unlink(missing_ok=True)` parametresini kullanmak `FileNotFoundError` istisnasını engeller.


In [ ]:
import os
from pathlib import Path

temp_dir: Path = Path('temp_workspace')
temp_dir.mkdir(exist_ok=True)

for idx in range(3):
    (temp_dir / f'artifact_{idx}.txt').write_text(f'Data {idx}', encoding='utf-8')

for filename in os.listdir(temp_dir):
    file_path: str = os.path.join(temp_dir, filename)
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f'Deleted file: {file_path}')

os.rmdir(temp_dir)
print(f'Directory exists: {temp_dir.exists()}')

for leftover in ('sample_read.txt', 'app_events.txt'):
    Path(leftover).unlink(missing_ok=True)


## 5. JSON Format Fundamentals

JSON (JavaScript Object Notation), dilden bağımsız, insan tarafından okunabilir ve özellikle web API'leri ile yapılandırma dosyalarında veri alışverişi için kullanılan standart metin tabanlı veri formatıdır.

**Python Sözlükleri ile JSON Sözdizimi Farkları**
* **Tırnak İşaretleri:** JSON standardında tüm anahtarlar (keys) ve metin değerleri **çift tırnak (`"`)** ile yazılmak zorundadır; tek tırnak (`'`) geçersizdir.
* **Veri Tipi Eşleşmeleri:** Python'daki `True` / `False` değerleri JSON'da küçük harfle `true` / `false`, `None` ise `null` olarak temsil edilir. JSON yalnızca metin tabanlı olduğu için doğrudan ayrıştırılmadan (`json.load` / `json.loads`) anahtar bazlı erişim yapılamaz.


In [ ]:
import json

raw_json: str = '{"service": "audio-engine", "active": true, "fallback": null, "channels": 2}'
parsed: dict[str, object] = json.loads(raw_json)

print(type(parsed))
print(parsed)
print(parsed['active'], parsed['fallback'])


## 6. Handling JSON (`load`, `loads`, `dump`, `dumps`)

Python'ın yerleşik `json` modülü, JSON verileri ile Python nesneleri (`dict`, `list`, `str`, `int`, `float`, `bool`, `None`) arasında çift yönlü serileştirme (serialization) ve seriden çıkarma (deserialization) sağlar.

**Syntax**
```python
import json

data = json.load(file_obj)
data = json.loads(json_str)

json.dump(obj, file_obj, indent=2)
json_str = json.dumps(obj, indent=2)
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Dosya vs. String Ayrımı:** Sonunda `s` harfi bulunan `loads()` ve `dumps()` fonksiyonları bellekteki `str` nesneleriyle; `load()` ve `dump()` ise doğrudan dosya nesneleriyle (`open(...)`) çalışır.
* **Okunabilirlik ve Karakter Kodlaması:** Dosyaya yazarken `indent=2` parametresi okunabilir girintili çıktı üretir; Türkçe veya Unicode karakterlerin `\uXXXX` kaçış dizilerine dönüşmemesi için `ensure_ascii=False` parametresi eklenmelidir.


In [ ]:
import json
from pathlib import Path

profile: dict[str, object] = {
    'user': 'Mustafa',
    'district': 'Kağıthane',
    'active': True,
    'secondary_node': None,
}

json_text: str = json.dumps(profile, indent=2, ensure_ascii=False)
print(json_text)

json_path: Path = Path('user_profile.json')
with open(json_path, mode='w', encoding='utf-8') as f:
    json.dump(profile, f, indent=2, ensure_ascii=False)

with open(json_path, mode='r', encoding='utf-8') as f:
    loaded_profile: dict[str, object] = json.load(f)

print(loaded_profile['district'])
json_path.unlink(missing_ok=True)


## 7. Caching API JSON Responses Locally

Dış API'lerden çekilen JSON verilerini yerel bir önbellek dosyasında saklamak, ağ gecikmesini ortadan kaldırır, çevrimdışı çalışmaya olanak tanır ve tekrarlayan API çağrılarının maliyetini/kota tüketimini düşürür.

**Mimari Akış ve Dikkat Edilmesi Gerekenler**
* **Önbellek Kontrolü:** `update=False` olduğunda önce yerel JSON dosyası okunmaya çalışılır; dosya yoksa (`FileNotFoundError`) veya bozuksa (`json.JSONDecodeError`) istisna yakalanarak ağ isteğine düşülür (fallback).
* **Hata Yönetimi:** HTTP isteklerinde zaman aşımı (`timeout`) ve durum kodu kontrolü (`response.raise_for_status()`) kullanmak bozuk veya hatalı yanıtların yerel önbelleğe yazılmasını önler.


In [ ]:
import json
from pathlib import Path
from urllib.request import urlopen


def fetch_cached_json(*, url: str, cache_path: Path, update: bool = False) -> dict[str, object]:
    cached_data: dict[str, object] | None = None

    if not update:
        try:
            with open(cache_path, mode='r', encoding='utf-8') as f:
                cached_data = json.load(f)
            print('Loaded data from local cache.')
        except (FileNotFoundError, json.JSONDecodeError) as err:
            print(f'Cache miss ({type(err).__name__}), fetching remote data...')

    if cached_data is None:
        with urlopen(url, timeout=10) as response:
            cached_data = json.loads(response.read().decode('utf-8'))
        with open(cache_path, mode='w', encoding='utf-8') as f:
            json.dump(cached_data, f, indent=2)
        print('Saved fresh response to local cache.')

    return cached_data


cache_file: Path = Path('comments_cache.json')
endpoint: str = 'https://dummyjson.com/comments?limit=2'

first_call: dict[str, object] = fetch_cached_json(url=endpoint, cache_path=cache_file)
second_call: dict[str, object] = fetch_cached_json(url=endpoint, cache_path=cache_file)
print(f'Total items in response: {len(second_call.get("comments", []))}')
cache_file.unlink(missing_ok=True)


## 8. Pattern Matching File Paths (`glob` & `iglob`)

`glob` modülü, Unix kabuk (shell) joker karakter kurallarını kullanarak belirtilen kalıpla eşleşen tüm dosya ve dizin yollarını bulmayı sağlar.

**Syntax**
```python
import glob

matches = glob.glob('**/*.py', root_dir='.', recursive=True, include_hidden=False)
lazy_matches = glob.iglob('**/*.py', recursive=True)
```

**Joker Karakter Kuralları ve Dikkat Edilmesi Gerekenler**
* **`*`:** Sıfır veya daha fazla karakterle eşleşir (`*.js`).
* **`?`:** Yalnızca tek bir karakterle eşleşir (`??dex.js`).
* **`[abc]` / `[!abc]`:** Köşeli parantez içindeki karakterlerden biriyle (veya `!` varsa o karakterler dışındaki biriyle) eşleşir.
* **`**` ve `recursive=True`:** Tüm alt dizinleri özyinelemeli olarak tarar. Python 3.11+ ile gelen `include_hidden=True` parametresi nokta (`.`) ile başlayan gizli dosya ve klasörleri de aramaya dahil eder.
* **Bellek Verimliliği:** `glob.glob()` tüm sonuçları tek seferde bir `list` olarak döndürürken, `glob.iglob()` sonuçları tek tek üreten bir yineleyici (iterator) döndürür.


In [ ]:
import glob
import shutil
from pathlib import Path

sandbox: Path = Path('glob_sandbox')
(sandbox / 'nested').mkdir(parents=True, exist_ok=True)

for rel in ('index.js', 'landing.js', 'sample.js', 'nested/helper.js'):
    (sandbox / rel).write_text('// js', encoding='utf-8')

print(glob.glob('??dex.js', root_dir=sandbox))
print(glob.glob('[il]*.js', root_dir=sandbox))
print(glob.glob('[!il]*.js', root_dir=sandbox))

lazy_iter = glob.iglob('**/*.js', root_dir=sandbox, recursive=True)
for idx, path in enumerate(lazy_iter, start=1):
    print(f'{idx}: {path}')

shutil.rmtree(sandbox)


## 9. Object Serialization (`pickle`)

`pickle` modülü, özel sınıf örnekleri (instances) dahil karmaşık Python nesne hiyerarşilerini ikili bir bayt akışına (byte stream) dönüştürmeyi (pickling) ve daha sonra aynı durumlarıyla (state) geri yüklemeyi (unpickling) sağlar.

**Syntax**
```python
import pickle

with open('state.pkl', mode='wb') as f:
    pickle.dump(obj, f)

with open('state.pkl', mode='rb') as f:
    restored_obj = pickle.load(f)
```

**Kullanım Alanları ve Kritik Güvenlik Uyarısı**
* **Ne Zaman Kullanılır:** JSON'a doğrudan dönüştürülemeyen özel sınıf örneklerini, makine öğrenmesi modellerini veya uygulama durumlarını ikili formatta (`'wb'` / `'rb'`) hızlıca kaydetmek için kullanılır.
* **Güvenlik Riski (Arbitrary Code Execution):** `pickle` modülü **güvenli değildir**. Seriden çıkarma (`pickle.load` / `pickle.loads`) işlemi sırasında kötü niyetli hazırlanmış bir `.pkl` dosyası sisteminizde doğrudan rastgele Python kodu çalıştırabilir. Güvenilmeyen veya dış kaynaklardan gelen pickle dosyaları asla açılmamalıdır.


In [ ]:
import pickle
from pathlib import Path


class AudioProject:
    def __init__(self, title: str, sample_rate: int) -> None:
        self.title = title
        self.sample_rate = sample_rate

    def summary(self) -> str:
        return f'{self.title} @ {self.sample_rate} Hz'


project: AudioProject = AudioProject('Flamenco Fusion', 44100)
project.sample_rate = 48000

pickle_path: Path = Path('project_state.pkl')
with open(pickle_path, mode='wb') as f:
    pickle.dump(project, f)

with open(pickle_path, mode='rb') as f:
    restored: AudioProject = pickle.load(f)

print(restored.summary())
restored.sample_rate = 96000
print(restored.summary())
pickle_path.unlink(missing_ok=True)
